# Text2Invest

Convert selected text into structured investment ideas using LLM-powered analysis.

**Pipeline stages:**
1. Extraction - Clean and normalize text
2. Entity - Identify companies, products, macro terms
3. Ticker - Map companies to stock symbols
4. Thesis - Generate investment thesis
5. Critique - Analyze risks and counter-thesis
6. Confidence - Calibrate confidence score
7. Formatter - Create final report

In [ ]:
!pip install -q litellm pydantic

## Configuration

Set your LLM provider settings below.

In [ ]:
# @title LLM Configuration { run: "auto" }
PROVIDER = "openai"  # @param ["openai", "anthropic", "ollama"]
MODEL = "gpt-4o"  # @param {type:"string"}
API_KEY = ""  # @param {type:"string"}
BASE_URL = ""  # @param {type:"string"}
TEMPERATURE = 0.7  # @param {type:"slider", min:0, max:2, step:0.1}

## Models & Types

In [ ]:
import json
from abc import ABC, abstractmethod
from datetime import datetime
from enum import Enum
from typing import Any, TypeVar
from uuid import UUID, uuid4

from litellm import acompletion
from pydantic import BaseModel, Field, ValidationError, field_validator


class Horizon(str, Enum):
    SHORT = "short"
    MEDIUM = "medium"
    LONG = "long"


class Provider(str, Enum):
    OPENAI = "openai"
    ANTHROPIC = "anthropic"
    OLLAMA = "ollama"


class Source(BaseModel):
    url: str = Field(..., description="Page URL")
    title: str = Field(..., description="Page title")


class Ticker(BaseModel):
    symbol: str = Field(..., pattern=r"^[A-Z]{1,5}$", description="Stock ticker symbol")
    company_name: str = Field(..., description="Full company name")
    confidence: float = Field(..., ge=0, le=1, description="Mapping certainty")


class RationaleQuote(BaseModel):
    quote: str = Field(..., description="Exact text from selection")
    start_offset: int = Field(..., ge=0, description="Character offset from start")
    end_offset: int = Field(..., ge=1, description="Character offset to end")


class ProviderMeta(BaseModel):
    provider: Provider = Field(..., description="LLM provider name")
    model: str = Field(..., description="Model identifier")
    temperature: float = Field(..., ge=0, le=2, description="Generation temperature")
    pipeline_duration_ms: int = Field(..., ge=0, description="Total processing time in ms")


class IdeaReport(BaseModel):
    id: UUID = Field(..., description="Unique identifier")
    created_at: datetime = Field(..., description="Creation timestamp")
    source: Source = Field(..., description="Origin webpage information")
    selection_text: str = Field(..., min_length=20, max_length=8000, description="Original selected text")
    tickers: list[Ticker] = Field(default_factory=list, description="Identified securities")
    thesis: str = Field(..., description="Investment thesis statement")
    executive_summary: list[str] = Field(..., min_length=1, max_length=3, description="Max 3 bullet points")
    rationale_quotes: list[RationaleQuote] = Field(default_factory=list, description="Supporting text excerpts")
    catalysts: list[str] = Field(default_factory=list, description="Potential positive triggers")
    risks: list[str] = Field(default_factory=list, description="Identified risk factors")
    counter_thesis: str = Field(..., description="Opposing viewpoint")
    horizon: Horizon = Field(..., description="Time horizon")
    confidence_score: float = Field(..., ge=0, le=1, description="Overall confidence")
    confidence_explanation: str = Field(..., description="Why this confidence level")
    limitations: list[str] = Field(default_factory=list, description="Known limitations")
    provider_meta: ProviderMeta = Field(..., description="LLM provider information")


class UserSettings(BaseModel):
    provider: Provider = Field(..., description="LLM provider")
    model: str = Field(..., description="Model identifier")
    api_key: str | None = Field(None, description="API key for cloud providers")
    base_url: str | None = Field(None, description="Base URL for custom endpoints")
    temperature: float = Field(0.7, ge=0, le=2, description="Generation temperature")
    pii_redaction: bool = Field(True, description="Redact PII")
    web_lookup: bool = Field(False, description="Allow web searches")


# Agent output models
class ExtractionOutput(BaseModel):
    cleaned_text: str = Field(..., description="Normalized text")
    language: str = Field(..., description="Detected language code")
    word_count: int = Field(..., ge=0, description="Token/word count")


class EntityOutput(BaseModel):
    companies: list[str] = Field(default_factory=list, description="Company names mentioned")
    products: list[str] = Field(default_factory=list, description="Product names mentioned")
    macro_terms: list[str] = Field(default_factory=list, description="Economic/macro terms")


class TickerMapping(BaseModel):
    company: str = Field(..., description="Company name from EntityOutput")
    symbol: str = Field(..., pattern=r"^[A-Z]{1,5}$", description="Resolved ticker symbol")
    confidence: float = Field(..., ge=0, le=1, description="Mapping confidence")
    reasoning: str = Field(..., description="Why this mapping")


class TickerOutput(BaseModel):
    mappings: list[TickerMapping] = Field(default_factory=list, description="Company-to-ticker mappings")


class ThesisOutput(BaseModel):
    thesis: str = Field(..., description="Investment thesis")
    supporting_quotes: list[RationaleQuote] = Field(default_factory=list, description="Text evidence")
    catalysts: list[str] = Field(default_factory=list, description="Positive triggers")
    horizon: Horizon = Field(..., description="Time horizon")


class CritiqueOutput(BaseModel):
    risks: list[str] = Field(default_factory=list, description="Risk factors")
    counter_thesis: str = Field(..., description="Opposing view")
    missing_info: list[str] = Field(default_factory=list, description="What's not in the text")


class ConfidenceOutput(BaseModel):
    score: float = Field(..., ge=0, le=1, description="Confidence score 0.0 to 1.0")
    explanation: str = Field(..., description="Calibration reasoning")
    limitations: list[str] = Field(default_factory=list, description="Known weaknesses")

## LLM Provider

In [ ]:
class LLMProvider:
    def __init__(self, settings: UserSettings):
        self.settings = settings
        self._model_name = self._build_model_name()

    def _build_model_name(self) -> str:
        if self.settings.provider == Provider.OPENAI:
            if self.settings.base_url:
                return f"openai/{self.settings.model}"
            return self.settings.model
        elif self.settings.provider == Provider.ANTHROPIC:
            return f"anthropic/{self.settings.model}"
        elif self.settings.provider == Provider.OLLAMA:
            return f"ollama/{self.settings.model}"
        else:
            raise ValueError(f"Unknown provider: {self.settings.provider}")

    async def complete(
        self,
        prompt: str,
        system_prompt: str | None = None,
        response_format: type[BaseModel] | None = None,
    ) -> str:
        messages: list[dict[str, str]] = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})

        kwargs: dict[str, Any] = {
            "model": self._model_name,
            "messages": messages,
            "temperature": self.settings.temperature,
        }

        if self.settings.provider == Provider.OPENAI:
            if self.settings.api_key:
                kwargs["api_key"] = self.settings.api_key
            elif self.settings.base_url:
                kwargs["api_key"] = "not-needed"
            if self.settings.base_url:
                kwargs["api_base"] = self.settings.base_url
        elif self.settings.provider == Provider.ANTHROPIC and self.settings.api_key:
            kwargs["api_key"] = self.settings.api_key
        elif self.settings.provider == Provider.OLLAMA and self.settings.base_url:
            kwargs["api_base"] = self.settings.base_url

        if response_format:
            kwargs["response_format"] = {"type": "json_object"}
            if system_prompt:
                messages[0]["content"] += (
                    f"\n\nYou MUST respond with valid JSON matching this schema:\n"
                    f"{json.dumps(response_format.model_json_schema(), indent=2)}"
                )

        response = await acompletion(**kwargs)
        content = response.choices[0].message.content
        if content is None:
            raise ValueError("LLM returned empty response")
        return content

## Agents

In [ ]:
T = TypeVar("T", bound=BaseModel)


class BaseAgent(ABC):
    name: str = "base"
    max_retries: int = 3

    def __init__(self, settings: UserSettings):
        self.settings = settings
        self.llm = LLMProvider(settings)

    @abstractmethod
    def get_system_prompt(self) -> str:
        pass

    @abstractmethod
    def get_user_prompt(self, **kwargs) -> str:
        pass

    @abstractmethod
    def get_output_model(self) -> type[T]:
        pass

    async def run(self, **kwargs) -> T:
        system_prompt = self.get_system_prompt()
        user_prompt = self.get_user_prompt(**kwargs)
        output_model = self.get_output_model()

        for attempt in range(self.max_retries):
            try:
                response = await self.llm.complete(
                    prompt=user_prompt,
                    system_prompt=system_prompt,
                    response_format=output_model,
                )
                data = json.loads(response)
                return output_model.model_validate(data)
            except (json.JSONDecodeError, ValidationError) as e:
                if attempt == self.max_retries - 1:
                    raise ValueError(f"Agent {self.name} failed after {self.max_retries} retries: {e}")
                continue
        raise ValueError(f"Agent {self.name} failed unexpectedly")


class ExtractionAgent(BaseAgent):
    name = "extraction"

    def get_system_prompt(self) -> str:
        return """You are a text extraction specialist. Your job is to clean and normalize raw text for investment analysis.

You must:
1. Remove excessive whitespace, special characters, and formatting artifacts
2. Preserve the semantic meaning and key financial terms
3. Detect the language of the text
4. Count the words/tokens

Output valid JSON matching the schema."""

    def get_user_prompt(self, text: str, **_kwargs) -> str:
        return f"""Clean and normalize the following text for investment analysis:

---
{text}
---

Return cleaned_text, language code (e.g., "en"), and word_count."""

    def get_output_model(self) -> type[ExtractionOutput]:
        return ExtractionOutput


class EntityAgent(BaseAgent):
    name = "entity"

    def get_system_prompt(self) -> str:
        return """You are an entity extraction specialist for investment analysis.

Your job is to identify:
1. Company names mentioned in the text
2. Product names or services
3. Macro-economic terms and indicators

Be comprehensive but precise. Only extract entities clearly present in the text.
Do not infer or guess entities not mentioned.

Output valid JSON matching the schema."""

    def get_user_prompt(self, cleaned_text: str, **_kwargs) -> str:
        return f"""Extract all investment-relevant entities from this text:

---
{cleaned_text}
---

Return:
- companies: List of company names
- products: List of product/service names
- macro_terms: List of economic/macro terms (e.g., "inflation", "GDP", "interest rates")"""

    def get_output_model(self) -> type[EntityOutput]:
        return EntityOutput


class TickerAgent(BaseAgent):
    name = "ticker"

    def get_system_prompt(self) -> str:
        return """You are a stock ticker mapping specialist.

Your job is to map company names to their stock ticker symbols.

Rules:
1. Only map companies you are confident about
2. Use US stock exchanges (NYSE, NASDAQ) when possible
3. If a company is private or you're unsure, do not include it
4. Provide confidence scores:
   - 0.9-1.0: Well-known public company with clear ticker
   - 0.7-0.9: Likely correct but some ambiguity
   - 0.5-0.7: Uncertain, multiple possibilities
   - Below 0.5: Do not include
5. Explain your reasoning for each mapping

Output valid JSON matching the schema."""

    def get_user_prompt(self, companies: list[str], **_kwargs) -> str:
        if not companies:
            return "No companies to map. Return empty mappings list."
        companies_str = "\n".join(f"- {c}" for c in companies)
        return f"""Map these companies to their stock ticker symbols:

{companies_str}

For each company, provide:
- company: The original company name
- symbol: The ticker symbol (uppercase, 1-5 chars)
- confidence: Your confidence (0.0-1.0)
- reasoning: Why you chose this ticker"""

    def get_output_model(self) -> type[TickerOutput]:
        return TickerOutput


class ThesisAgent(BaseAgent):
    name = "thesis"

    def get_system_prompt(self) -> str:
        return """You are an investment thesis generator.

Your job is to create a structured investment thesis GROUNDED STRICTLY in the provided text.

Rules:
1. Your thesis must be directly supported by quotes from the text
2. Do not make claims not supported by the text
3. Identify catalysts (positive triggers) mentioned or implied
4. Determine an appropriate time horizon:
   - "short": Days to weeks
   - "medium": Months to a year
   - "long": Multiple years
5. For each supporting quote, provide exact character offsets

Output valid JSON matching the schema."""

    def get_user_prompt(self, cleaned_text: str, companies: list[str], tickers: list[str], **_kwargs) -> str:
        context = f"Companies: {', '.join(companies)}" if companies else "No specific companies"
        ticker_str = f"Tickers: {', '.join(tickers)}" if tickers else "No tickers identified"
        return f"""Generate an investment thesis based on this text:

---
{cleaned_text}
---

{context}
{ticker_str}

Your thesis must:
1. Be grounded in the text (include supporting_quotes with exact quotes and offsets)
2. Identify catalysts mentioned
3. Determine appropriate time horizon

Remember: Only claim what the text supports."""

    def get_output_model(self) -> type[ThesisOutput]:
        return ThesisOutput


class CritiqueAgent(BaseAgent):
    name = "critique"

    def get_system_prompt(self) -> str:
        return """You are an investment thesis critic and risk analyst.

Your job is to critically evaluate an investment thesis and identify:
1. Risks and potential downsides
2. A counter-thesis (the bear case)
3. Missing information that would strengthen or weaken the thesis

Rules:
1. Be thorough but fair
2. Focus on material risks, not trivial concerns
3. Consider market, company-specific, and macro risks
4. The counter-thesis should be a plausible alternative view
5. Missing info should highlight gaps in the analysis

Output valid JSON matching the schema."""

    def get_user_prompt(self, thesis: str, cleaned_text: str, companies: list[str], **_kwargs) -> str:
        context = f"Companies: {', '.join(companies)}" if companies else "No specific companies"
        return f"""Critique this investment thesis:

THESIS:
{thesis}

ORIGINAL TEXT:
{cleaned_text}

{context}

Provide:
1. risks: Material risk factors
2. counter_thesis: The opposing investment view
3. missing_info: What information is missing that would help evaluate this thesis"""

    def get_output_model(self) -> type[CritiqueOutput]:
        return CritiqueOutput


class ConfidenceAgent(BaseAgent):
    name = "confidence"

    def get_system_prompt(self) -> str:
        return """You are a confidence calibration specialist for investment theses.

Your job is to assign a well-calibrated confidence score to an investment thesis.

Calibration guidelines:
- 0.8-1.0: Strong thesis with clear evidence, identified company, and actionable catalysts
- 0.6-0.8: Reasonable thesis with some uncertainty or missing information
- 0.4-0.6: Speculative thesis with significant gaps or ambiguity
- 0.2-0.4: Weak thesis with little supporting evidence
- 0.0-0.2: Very weak or contradictory thesis

Consider:
1. Quality and specificity of source text
2. Clarity of company/ticker identification
3. Strength of supporting evidence
4. Presence of clear catalysts
5. Time horizon appropriateness
6. Identified risks and counter-thesis strength

Output valid JSON matching the schema."""

    def get_user_prompt(self, thesis: str, risks: list[str], counter_thesis: str, missing_info: list[str], tickers: list[dict], **_kwargs) -> str:
        ticker_str = ", ".join(f"{t['symbol']} ({t['confidence']:.2f})" for t in tickers) if tickers else "None identified"
        risks_str = "\n".join(f"- {r}" for r in risks) if risks else "None identified"
        missing_str = "\n".join(f"- {m}" for m in missing_info) if missing_info else "None"
        return f"""Calibrate confidence for this investment thesis:

THESIS:
{thesis}

TICKERS: {ticker_str}

RISKS:
{risks_str}

COUNTER-THESIS:
{counter_thesis}

MISSING INFORMATION:
{missing_str}

Provide:
1. score: Your calibrated confidence (0.0-1.0)
2. explanation: Why this confidence level
3. limitations: Known weaknesses in the analysis"""

    def get_output_model(self) -> type[ConfidenceOutput]:
        return ConfidenceOutput


class FormatterAgent:
    name = "formatter"

    def __init__(self, settings: UserSettings, pipeline_start: datetime):
        self.settings = settings
        self.pipeline_start = pipeline_start

    def format(
        self,
        selection_text: str,
        url: str,
        title: str,
        tickers: list[dict],
        thesis: str,
        supporting_quotes: list[dict],
        catalysts: list[str],
        horizon: str,
        risks: list[str],
        counter_thesis: str,
        confidence_score: float,
        confidence_explanation: str,
        limitations: list[str],
    ) -> IdeaReport:
        pipeline_duration_ms = int((datetime.now() - self.pipeline_start).total_seconds() * 1000)
        formatted_tickers = [
            Ticker(symbol=t["symbol"], company_name=t.get("company", t["symbol"]), confidence=t["confidence"])
            for t in tickers
        ]
        formatted_quotes = [
            RationaleQuote(quote=q["quote"], start_offset=q["start_offset"], end_offset=q["end_offset"])
            for q in supporting_quotes
        ]
        executive_summary = self._generate_executive_summary(thesis, formatted_tickers, confidence_score)
        return IdeaReport(
            id=uuid4(),
            created_at=datetime.now(),
            source=Source(url=url, title=title),
            selection_text=selection_text,
            tickers=formatted_tickers,
            thesis=thesis,
            executive_summary=executive_summary,
            rationale_quotes=formatted_quotes,
            catalysts=catalysts,
            risks=risks,
            counter_thesis=counter_thesis,
            horizon=Horizon(horizon),
            confidence_score=confidence_score,
            confidence_explanation=confidence_explanation,
            limitations=limitations,
            provider_meta=ProviderMeta(
                provider=self.settings.provider,
                model=self.settings.model,
                temperature=self.settings.temperature,
                pipeline_duration_ms=pipeline_duration_ms,
            ),
        )

    def _generate_executive_summary(self, thesis: str, tickers: list[Ticker], confidence: float) -> list[str]:
        summary = []
        if tickers:
            ticker_str = ", ".join(t.symbol for t in tickers[:3])
            summary.append(f"Tickers: {ticker_str}")
        thesis_short = thesis[:150] + "..." if len(thesis) > 150 else thesis
        summary.append(thesis_short)
        summary.append(f"Confidence: {confidence:.0%}")
        return summary[:3]

## Pipeline

In [ ]:
class Pipeline:
    def __init__(self, settings: UserSettings):
        self.settings = settings

    async def run(
        self,
        selection_text: str,
        url: str = "colab://notebook",
        title: str = "Colab Notebook",
        on_stage: callable = None,
    ) -> IdeaReport:
        pipeline_start = datetime.now()

        if on_stage:
            on_stage("extraction")
        extraction_agent = ExtractionAgent(self.settings)
        extraction_output = await extraction_agent.run(text=selection_text)
        print(f"Extraction: {extraction_output.word_count} words, language={extraction_output.language}")

        if on_stage:
            on_stage("entity")
        entity_agent = EntityAgent(self.settings)
        entity_output = await entity_agent.run(cleaned_text=extraction_output.cleaned_text)
        print(f"Entities: {len(entity_output.companies)} companies, {len(entity_output.products)} products")

        if on_stage:
            on_stage("ticker")
        ticker_agent = TickerAgent(self.settings)
        ticker_output = await ticker_agent.run(companies=entity_output.companies)
        print(f"Tickers: {[t.symbol for t in ticker_output.mappings]}")

        tickers_for_thesis = [t.symbol for t in ticker_output.mappings]

        if on_stage:
            on_stage("thesis")
        thesis_agent = ThesisAgent(self.settings)
        thesis_output = await thesis_agent.run(
            cleaned_text=extraction_output.cleaned_text,
            companies=entity_output.companies,
            tickers=tickers_for_thesis,
        )
        print(f"Thesis generated, horizon={thesis_output.horizon.value}")

        if on_stage:
            on_stage("critique")
        critique_agent = CritiqueAgent(self.settings)
        critique_output = await critique_agent.run(
            thesis=thesis_output.thesis,
            cleaned_text=extraction_output.cleaned_text,
            companies=entity_output.companies,
        )
        print(f"Critique: {len(critique_output.risks)} risks identified")

        tickers_for_confidence = [{"symbol": t.symbol, "confidence": t.confidence} for t in ticker_output.mappings]

        if on_stage:
            on_stage("confidence")
        confidence_agent = ConfidenceAgent(self.settings)
        confidence_output = await confidence_agent.run(
            thesis=thesis_output.thesis,
            risks=critique_output.risks,
            counter_thesis=critique_output.counter_thesis,
            missing_info=critique_output.missing_info,
            tickers=tickers_for_confidence,
        )
        print(f"Confidence: {confidence_output.score:.0%}")

        if on_stage:
            on_stage("formatting")
        formatter = FormatterAgent(self.settings, pipeline_start)

        tickers_for_formatter = [{"symbol": t.symbol, "company": t.company, "confidence": t.confidence} for t in ticker_output.mappings]
        quotes_for_formatter = [{"quote": q.quote, "start_offset": q.start_offset, "end_offset": q.end_offset} for q in thesis_output.supporting_quotes]

        return formatter.format(
            selection_text=selection_text,
            url=url,
            title=title,
            tickers=tickers_for_formatter,
            thesis=thesis_output.thesis,
            supporting_quotes=quotes_for_formatter,
            catalysts=thesis_output.catalysts,
            horizon=thesis_output.horizon.value,
            risks=critique_output.risks,
            counter_thesis=critique_output.counter_thesis,
            confidence_score=confidence_output.score,
            confidence_explanation=confidence_output.explanation,
            limitations=confidence_output.limitations,
        )

## Run Analysis

Paste the text you want to analyze below and run the cell.

In [ ]:
# @title Input Text { display-mode: "form" }
INPUT_TEXT = """Apple reported record Q1 2024 earnings with revenue of $119.6 billion, up 2% year-over-year. iPhone revenue was $69.7 billion, Services revenue reached an all-time high of $23.1 billion. CEO Tim Cook highlighted strong performance in emerging markets and the upcoming Vision Pro launch. The company authorized an additional $110 billion for share repurchases and increased its quarterly dividend by 4%."""  # @param {type:"string"}

In [ ]:
import asyncio

settings = UserSettings(
    provider=Provider(PROVIDER),
    model=MODEL,
    api_key=API_KEY if API_KEY else None,
    base_url=BASE_URL if BASE_URL else None,
    temperature=TEMPERATURE,
)

pipeline = Pipeline(settings)
report = await pipeline.run(selection_text=INPUT_TEXT)

print("\n" + "="*60)
print("INVESTMENT IDEA REPORT")
print("="*60)

## View Report

In [ ]:
print(f"ID: {report.id}")
print(f"Created: {report.created_at}")
print(f"\nTickers: {', '.join(t.symbol for t in report.tickers)}")
print(f"\nThesis:\n{report.thesis}")
print(f"\nExecutive Summary:")
for point in report.executive_summary:
    print(f"  - {point}")
print(f"\nCatalysts:")
for c in report.catalysts:
    print(f"  - {c}")
print(f"\nRisks:")
for r in report.risks:
    print(f"  - {r}")
print(f"\nCounter-Thesis:\n{report.counter_thesis}")
print(f"\nHorizon: {report.horizon.value}")
print(f"Confidence: {report.confidence_score:.0%}")
print(f"\nConfidence Explanation:\n{report.confidence_explanation}")
print(f"\nLimitations:")
for l in report.limitations:
    print(f"  - {l}")
print(f"\nProcessing Time: {report.provider_meta.pipeline_duration_ms}ms")

In [ ]:
# Export as JSON
print(report.model_dump_json(indent=2))